# Lab 1.6 — RAG for C# Unit Test Generation

This notebook extends the Gemini unit test generator with ChromaDB.
Given a new C# input X, it retrieves the three most similar examples
from a collection of at least 1,000 C# code and xUnit test pairs.

Pipeline: input X → retrieve 3 examples → build the augmented prompt
→ generate xUnit tests y with Gemini.

In [2]:
%pip uninstall -y opentelemetry-exporter-otlp-common
%pip install "google-genai==2.12.1" "google-auth==2.49.0" "chromadb==1.5.9" "opentelemetry-api==1.42.1" "opentelemetry-sdk==1.42.1" "opentelemetry-exporter-otlp-proto-grpc==1.42.1" "kubernetes==35.0.0" "oauthlib==3.3.1"

Found existing installation: opentelemetry-exporter-otlp-common 0.66b2
Uninstalling opentelemetry-exporter-otlp-common-0.66b2:
  Successfully uninstalled opentelemetry-exporter-otlp-common-0.66b2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.9 MB/s  0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.45.2
    Uninstalling opentelemetry-proto-1.45.2:
      Successfully uninstalled opentelemetry-proto-1.45.2
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.45.2
    Uninstalling opentelemetry-api-1.45.2:
      Successfully uninstalled opentelemetry-api-1.45.2
  Attempting uninstall: oauthlib
    Found existing installation: oauthlib 4.0.1
    Uninstalling oauthlib-4.0.1:
      Successfully uninstalled oauthlib-4.0.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.66b2
    Uninstalling opentelemetry

In [3]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_KEY"))
model = "gemini-3.5-flash-lite"

print("Gemini client initialized.")

Gemini client initialized.


In [5]:
import chromadb

print("ChromaDB ready:", chromadb.__version__)

ChromaDB ready: 1.5.9


In [4]:
import json
from collections import Counter
from pathlib import Path


def make_pair(example_id, family, class_name, args, behavior, rule, cases):
    parameters = ", ".join(f"int {name}" for name in args)
    call_args = ", ".join(args)
    source = f"""namespace Lab16.Reference;

public static class {class_name}
{{
    public static bool IsAllowed({parameters})
    {{
        return {rule};
    }}
}}"""

    rows = []
    for *values, expected in cases:
        arguments = ", ".join(str(value) for value in values)
        rows.append(f"    [InlineData({arguments}, {str(expected).lower()})]")
    attributes = "\n".join(rows)

    tests = f"""using Lab16.Reference;
using Xunit;

public class {class_name}Tests
{{
    [Theory]
{attributes}
    public void IsAllowed_ReturnsExpected({parameters}, bool expected)
    {{
        Assert.Equal(expected, {class_name}.IsAllowed({call_args}));
    }}
}}"""
    return {
        "id": example_id,
        "family": family,
        "X": f"{behavior}\n\n{source}",
        "y": tests,
    }


examples = []
for i in range(1, 251):
    limit = i + 11
    examples.append(make_pair(
        f"stock-{i:03}", "stock", "PurchaseQuantityValidator",
        ("quantity", "availableStock"),
        f"Accept positive purchase quantities up to {limit} and within available stock.",
        f"quantity > 0 && quantity <= availableStock && quantity <= {limit}",
        [
            (1, limit, True),
            (5, 5, True),
            (limit, limit + 5, True),
            (0, limit, False),
            (-1, limit, False),
            (1, 0, False),
            (5, 4, False),
            (limit + 1, limit + 5, False),
        ],
    ))

    low = -40 + (i - 1) // 25
    high = 15 + (i - 1) % 25
    examples.append(make_pair(
        f"temperature-{i:03}", "temperature", "TemperatureValidator",
        ("temperature",),
        f"Accept temperatures from {low} through {high}, including both limits.",
        f"temperature >= {low} && temperature <= {high}",
        [(low, True), (high, True), (low - 1, False), (high + 1, False)],
    ))

    weight_limit = 1000 + 50 * i
    parcel_limit = 5 + i % 12
    examples.append(make_pair(
        f"shipping-{i:03}", "shipping", "ShippingValidator",
        ("weightGrams", "parcelCount"),
        f"Accept positive shipment weights through {weight_limit} g "
        f"and positive parcel counts through {parcel_limit}.",
        f"weightGrams > 0 && weightGrams <= {weight_limit} "
        f"&& parcelCount > 0 && parcelCount <= {parcel_limit}",
        [
            (weight_limit, parcel_limit, True),
            (weight_limit + 1, 1, False),
            (1, parcel_limit + 1, False),
            (0, 1, False),
            (1, 0, False),
        ],
    ))

    pack = 2 + (i - 1) % 10
    cap = pack * (2 + (i - 1) // 10)
    examples.append(make_pair(
        f"pack-{i:03}", "pack", "PackSizeValidator",
        ("itemCount",),
        f"Accept positive item counts up to {cap} that are divisible by pack size {pack}.",
        f"itemCount > 0 && itemCount <= {cap} && itemCount % {pack} == 0",
        [(pack, True), (cap, True), (pack + 1, False), (cap + pack, False), (0, False)],
    ))

assert len(examples) == len({example["id"] for example in examples}) == 1000
assert len({example["X"] for example in examples}) == 1000

dataset_path = Path("examples.jsonl")
with dataset_path.open("w", encoding="utf-8") as dataset:
    for example in examples:
        dataset.write(json.dumps(example) + "\n")

print("Total pairs", len(examples))
print("Pairs by family", dict(Counter(example["family"] for example in examples)))

Total pairs 1000
Pairs by family {'stock': 250, 'temperature': 250, 'shipping': 250, 'pack': 250}


In [6]:
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

chroma = chromadb.PersistentClient(path="lab16_chroma")
collection = chroma.get_or_create_collection(
    name="lab16-csharp-examples",
    embedding_function=DefaultEmbeddingFunction(),
    configuration={"hnsw": {"space": "cosine"}},
)

for start in range(0, len(examples), 100):
    batch = examples[start:start + 100]
    collection.upsert(
        ids=[example["id"] for example in batch],
        documents=[example["X"] for example in batch],
        metadatas=[
            {"family": example["family"], "y": example["y"]}
            for example in batch
        ],
    )

print("Pairs stored in ChromaDB", collection.count())
assert collection.count() == len(examples)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:05<00:00, 15.0MiB/s]


Pairs stored in ChromaDB 1000


In [7]:
def retrieve_examples(new_x):
    hits = collection.query(
        query_texts=[new_x],
        n_results=3,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {"id": example_id, "X": x, "y": meta["y"], "distance": distance}
        for example_id, x, meta, distance in zip(
            hits["ids"][0], hits["documents"][0],
            hits["metadatas"][0], hits["distances"][0],
        )
    ]


new_x = """An order quantity must be positive, no greater than available stock,
and at most 10 items. A quantity equal to either upper limit is allowed.

namespace Lab14.Target;

public static class OrderQuantityValidator
{
    public const int MaxPerOrder = 10;

    public static bool IsAllowed(int quantity, int availableStock)
    {
        return quantity > 0
            && quantity <= availableStock
            && quantity <= MaxPerOrder;
    }
}
"""

assert new_x not in {example["X"] for example in examples}

print("Input X\n")
print(new_x)

retrieved = retrieve_examples(new_x)
assert len(retrieved) == 3

print("Top 3 retrieved examples")
for rank, example in enumerate(retrieved, start=1):
    print(f"{rank}. {example['id']} — cosine distance {example['distance']:.4f}")

Input X

An order quantity must be positive, no greater than available stock,
and at most 10 items. A quantity equal to either upper limit is allowed.

namespace Lab14.Target;

public static class OrderQuantityValidator
{
    public const int MaxPerOrder = 10;

    public static bool IsAllowed(int quantity, int availableStock)
    {
        return quantity > 0
            && quantity <= availableStock
            && quantity <= MaxPerOrder;
    }
}

Top 3 retrieved examples
1. stock-091 — cosine distance 0.1900
2. stock-103 — cosine distance 0.1911
3. stock-108 — cosine distance 0.1916


In [8]:
instructions = (
    "Generate one C# xUnit test file for the new observation. "
    "Verify valid inputs, invalid inputs, and boundary behavior. "
    "Use only the public API of the new class and import its namespace. "
    "Return the test code as plain text."
)

target = f"New observation X\n{new_x}"
without_rag = f"{instructions}\n\n{target}"

reference_blocks = []
for example in retrieved:
    reference_blocks.append(
        f"Reference {example['id']}\n"
        f"Example X\n{example['X']}\n\n"
        f"Example y\n{example['y']}"
    )

references = "\n\n".join(reference_blocks)
with_rag = f"{instructions}\n\n{references}\n\n{target}"

print("Augmented prompt sent to Gemini\n")
print(with_rag)

Augmented prompt sent to Gemini

Generate one C# xUnit test file for the new observation. Verify valid inputs, invalid inputs, and boundary behavior. Use only the public API of the new class and import its namespace. Return the test code as plain text.

Reference stock-091
Example X
Accept positive purchase quantities up to 102 and within available stock.

namespace Lab16.Reference;

public static class PurchaseQuantityValidator
{
    public static bool IsAllowed(int quantity, int availableStock)
    {
        return quantity > 0 && quantity <= availableStock && quantity <= 102;
    }
}

Example y
using Lab16.Reference;
using Xunit;

public class PurchaseQuantityValidatorTests
{
    [Theory]
    [InlineData(1, 102, true)]
    [InlineData(5, 5, true)]
    [InlineData(102, 107, true)]
    [InlineData(0, 102, false)]
    [InlineData(-1, 102, false)]
    [InlineData(1, 0, false)]
    [InlineData(5, 4, false)]
    [InlineData(103, 107, false)]
    public void IsAllowed_ReturnsExpected(int q

In [9]:
def generate_tests(prompt):
    response = client.models.generate_content(model=model, contents=prompt)
    if not response.text:
        raise RuntimeError("Gemini returned no text.")
    return response.text.strip()


results = {}
print("Gemini model", model)

for label, prompt in (("Without RAG", without_rag), ("With RAG", with_rag)):
    results[label] = generate_tests(prompt)
    print(f"\n=== {label} — output y ===\n")
    print(results[label])

Gemini model gemini-3.5-flash-lite

=== Without RAG — output y ===

using Xunit;
using Lab14.Target;

namespace Lab14.Tests;

public class OrderQuantityValidatorTests
{
    [Theory]
    [InlineData(1, 10)]   // Minimum valid boundary
    [InlineData(5, 10)]   // Mid-range valid
    [InlineData(10, 10)]  // Maximum valid per order (MaxPerOrder boundary)
    [InlineData(5, 5)]    // Maximum valid based on available stock boundary
    public void IsAllowed_ValidInputs_ReturnsTrue(int quantity, int availableStock)
    {
        bool result = OrderQuantityValidator.IsAllowed(quantity, availableStock);
        Assert.True(result);
    }

    [Theory]
    [InlineData(0, 10)]   // Below minimum positive boundary
    [InlineData(-1, 10)]  // Negative quantity
    [InlineData(11, 15)]  // Exceeds MaxPerOrder (10)
    [InlineData(6, 5)]    // Exceeds available stock
    [InlineData(1, 0)]    // Zero available stock
    [InlineData(1, -5)]   // Negative available stock
    public void IsAllowed_In

## Results comparison

Both runs used the same `OrderQuantityValidator` input and the same Gemini model. The augmented prompt included `stock-091`, `stock-103`, and `stock-108`.

| Scenario | Without RAG | With RAG |
| --- | --- | --- |
| Reject zero and negative quantities | Yes | Yes |
| Reject a positive quantity when stock is zero | Yes | Yes |
| Accept a quantity equal to stock below the 10-item limit | Yes | Yes |
| Reject a quantity above available stock | Yes | Yes |
| Accept 10 items when more than 10 are in stock | No | Yes `(10, 15)` |
| Reject 11 items when enough stock is available | Yes | Yes |

The run without RAG generated 10 cases and covered 5 of the 6 selected scenarios. The RAG run generated 9 cases and covered all 6, including the 10-item limit with stock to spare. The run without RAG also checked a negative stock value. These results come from inspection of the generated code for this one input.
